# 15 - Contraste de hipótesis
En este notebook contrastamos H1, H2 y H5 con la matriz de experimentos. Seguimos el criterio de la memoria:
- Una hipótesis se refuta en un caso cuando la F1 macro del conjunto de variables que se evalúa es inferior al 80 % de la del conjunto `completo` (`CLEARLY_INFERIOR_RATIO`), con el mismo modelo, la misma estrategia y el mismo perfil.
- La matriz con los hiperparámetros ajustados es la principal y la de los valores por defecto sirve de comprobación.
- Una conclusión es robusta si coincide en los cuatro modelos (el dummy es solo línea base y no se usa).

Cada hipótesis tiene 12 casos, 3 estrategias por 4 perfiles.

## Carga de los resultados

In [ ]:
import pandas as pd

from urban_accessibility_ml.config import MATRIX_MODELS, REPORTS_DIR
from urban_accessibility_ml.modeling.evaluate import cases_below, summarize_folds

CASE = ["strategy", "feature_set", "profile"]
MODELS = [m for m in MATRIX_MODELS if m != "dummy"]
# Conjunto que se evalúa en cada hipótesis (tab:comparaciones-hipotesis de la memoria)
HYPOTHESES = {"H1": "cartografico_aereo", "H2": "calle", "H5": "combinado"}

summary = {
    name: summarize_folds(pd.read_csv(REPORTS_DIR / "results" / f"experiment-matrix-{name}.csv"), CASE + ["model"])
    for name in ["tuned", "default"]
}
summary["tuned"].shape, summary["default"].shape

## Casos por debajo del 80 % de completo
Número de casos de 12 en los que cada hipótesis queda por debajo del 80 % de completo, por modelo y con cada matriz.
- H1 se evalúa con `cartografico_aereo`.
- H2 con `calle`.
- H5 con `combinado`.

In [ ]:
def count_below(summary):
    counts = pd.DataFrame({m: cases_below(summary, m)["feature_set"].value_counts() for m in MODELS})
    return counts.reindex(list(HYPOTHESES.values())).fillna(0).astype(int).set_axis(list(HYPOTHESES))


display(count_below(summary["tuned"]).rename_axis("ajustada"))
display(count_below(summary["default"]).rename_axis("por defecto"))

Observamos que la hipótesis 1, sobre cartográfico y aéreo:
- En la regresión logística, 9 de los 12 casos de la matriz ajustada quedan por debajo del 80% de completo, mientras que sin ajustar son 4 de 12.
- Se comporta mejor en random_forest, XGBoost o LightGBM, con 3 o menos casos tanto en ajustada como por defecto.

La hipótesis 2, sobre datos del conjunto a pie de calle:
- En ajustada, la regresión logística tiene 3 casos de 12 por debajo del 80%, así como 1 caso de LightGBM.
- Mientras que en por defecto, únicamente 1 caso de LightGBM está por debajo del 80%.

En cuanto a la hipótesis 5, sobre ambos conjuntos de imágenes, se repiten los mismos valores que a pie de calle.
Como esta comprobación se ha hecho con el umbral del 50%, recordemos que el conjunto de calle y combinado tenía las mismas dimensiones.